# Day 40: Integrate Conversational Memory

Welcome to Day 40! Today, we are focusing on a critical component of building Agentic AI: **Conversational Memory**.

## Core Theory (Just-in-Time)

**Why Memory?**
By default, Large Language Models (LLMs) and standard LLM chains are **stateless**. They do not remember previous interactions. If you ask an LLM "What is my name?" and then in the next request ask "What did I just ask you?", it will have no idea.
In production, for chatbots or agents to be useful, they need context about the ongoing conversation.

**How does Memory work?**
At a high level, memory works by intercepting the user's input, appending the history of previous messages to the prompt, and sending the combined prompt to the LLM. The LLM's response is then saved back into the memory for the next turn.

**Types of Memory in LangChain:**
1.  **`ConversationBufferMemory`**: Stores the raw, complete transcript of the conversation. *Pros:* Perfect recall. *Cons:* Context window fills up quickly, leading to higher token costs and potential "Lost in the Middle" issues.
2.  **`ConversationBufferWindowMemory`**: Keeps a sliding window of the last *k* interactions. *Pros:* Bounded token usage. *Cons:* Forgets older details (like the user's name mentioned at the start).
3.  **`ConversationSummaryMemory`**: Uses an LLM to dynamically summarize the conversation as it happens. *Pros:* Long-term context with small token footprint. *Cons:* Requires extra LLM calls (latency/cost) and might lose fine-grained details during summarization.

Today, we'll focus on modern LCEL (LangChain Expression Language) implementations for memory, specifically `RunnableWithMessageHistory`, alongside buffer concepts and summarization concepts.
**AI Security Implications:**
- **PII Leakage:** Storing raw conversational history in memory or a database introduces significant PII risks. Sensitive data should be redacted or masked before being appended to the session history to prevent it from leaking into logs, analytics, or subsequent LLM calls.
- **Prompt Injection via Memory:** Malicious users might inject harmful instructions into their early messages. If these messages are persisted in memory and fed back into the prompt unescaped, they can override system instructions. Validate and sanitize inputs before saving context.
- **Fallback Mechanisms:** In production, retrieving session history from external stores (like Redis) can fail. Always implement fallback mechanisms (e.g., initializing an empty, ephemeral history) to ensure the chat session degrades gracefully instead of crashing.


## Code Implementation

Let's look at how to implement conversational memory using the modern `RunnableWithMessageHistory` abstraction in LangChain.

### Basic Implementation

Isolates the core concept of `RunnableWithMessageHistory` bounding memory size using list slicing in a custom history getter.

In [ ]:
import os
from typing import Dict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

store: Dict[str, BaseChatMessageHistory] = {}

def get_session_history_windowed(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    # For a basic window buffer effect, just ensure the list doesn't grow indefinitely
    history = store[session_id]
    if len(history.messages) > 4:
        history.messages = history.messages[-4:]
    return history

def basic_window_memory_example() -> None:
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ])
    
    chain = prompt | llm
    with_message_history = RunnableWithMessageHistory(
        chain,
        get_session_history_windowed,
        input_messages_key="question",
        history_messages_key="history",
    )
    
    config = {"configurable": {"session_id": "basic_window_user"}}
    try:
        res = with_message_history.invoke({"question": "Hi, I'm Alice!"}, config=config)
        print("AI:", res.content)
    except Exception as e:
        print(f"API Error (expected without valid key): {e}")

basic_window_memory_example()


### Medium Implementation

Demonstrates `ConversationSummaryMemory` to compress long chat history dynamically, integrated into a clean OOP structure.

In [ ]:
from langchain.memory import ConversationSummaryMemory
from langchain_core.messages import SystemMessage

class SummaryMemoryManager:
    """Manages summary-based conversational history states cleanly."""
    
    def __init__(self):
        self.summary_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        self._store: Dict[str, ConversationSummaryMemory] = {}
        
    def get_history(self, session_id: str) -> BaseChatMessageHistory:
        if session_id not in self._store:
            self._store[session_id] = ConversationSummaryMemory(
                llm=self.summary_llm, return_messages=True
            )
        
        # A trick for LCEL: we return a ChatMessageHistory that contains the summarized buffer
        # so that RunnableWithMessageHistory can append to it.
        memory_obj = self._store[session_id]
        
        # Load current variables
        variables = memory_obj.load_memory_variables({})
        summary = variables.get("history", "")
        
        temp_history = ChatMessageHistory()
        if summary:
            # Add summary as a system message to context
            if isinstance(summary, list):
                for msg in summary:
                    temp_history.add_message(msg)
            else:
                temp_history.add_message(SystemMessage(content=f"Summary of previous chat: {summary}"))
                
        # In a real implementation, you'd override `add_message` on temp_history to write back to the memory_obj.
        return temp_history

class SummarizingAgent:
    """Agent utilizing a provided SummaryMemoryManager."""
    
    def __init__(self, memory_manager: SummaryMemoryManager):
        self.memory_manager = memory_manager
        self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful AI."),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ])
        self.chain = self.prompt | self.llm
        self.chain_with_memory = RunnableWithMessageHistory(
            self.chain,
            self.memory_manager.get_history,
            input_messages_key="input",
            history_messages_key="chat_history",
        )
        
    def chat(self, session_id: str, user_input: str) -> str:
        config = {"configurable": {"session_id": session_id}}
        try:
            # In an actual setup with ConversationSummaryMemory in LCEL, 
            # you often need a custom chain or to manually save context after invoke.
            response = self.chain_with_memory.invoke({"input": user_input}, config=config)
            
            # Manually save context to the underlying ConversationSummaryMemory
            self.memory_manager._store.setdefault(session_id, ConversationSummaryMemory(llm=self.llm))
            self.memory_manager._store[session_id].save_context(
                {"input": user_input}, {"output": response.content}
            )
            return response.content
        except Exception as e:
            return f"Error communicating with LLM: {e}"

def medium_memory_example() -> None:
    manager = SummaryMemoryManager()
    agent = SummarizingAgent(manager)
    print("Medium Agent Response:", agent.chat("user_med_summary", "Hello, OOP Agent! I like Python."))
    
medium_memory_example()


### Advanced Implementation

Production-grade LCEL implementation with strict type hinting, docstrings, exact import syntax, and AI Security (PII Redaction before appending to memory, Fallbacks, strict window buffering).

In [ ]:
import re
import logging
from typing import Dict, Optional, Any
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecureMemoryManager(BaseChatMessageHistory):
    """A secure wrapper for ChatMessageHistory that redacts PII before saving messages, maintaining a bounded window length."""
    
    def __init__(self, k: int = 5):
        self.history = ChatMessageHistory()
        self.k = k

    def _redact_pii(self, text: str) -> str:
        email_pattern = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
        return re.sub(email_pattern, "[REDACTED_EMAIL]", text)
        
    def add_message(self, message: BaseMessage) -> None:
        if isinstance(message.content, str):
            message.content = self._redact_pii(message.content)
        self.history.add_message(message)
        
        # Implement manual windowing
        if len(self.history.messages) > self.k * 2:
            self.history.messages = self.history.messages[-(self.k * 2):]
            
    @property
    def messages(self):
        return self.history.messages

    def clear(self):
        self.history.clear()

class ProductionMemoryStore:
    """A robust memory store demonstrating fallback mechanisms."""
    def __init__(self) -> None:
        self._store: Dict[str, SecureMemoryManager] = {}
        
    def get_history(self, session_id: str) -> BaseChatMessageHistory:
        try:
            if session_id not in self._store:
                self._store[session_id] = SecureMemoryManager(k=3)
            return self._store[session_id]
        except Exception as e:
            logger.error(f"Failed to retrieve session {session_id}: {e}")
            logger.warning("Using ephemeral memory as fallback.")
            return SecureMemoryManager(k=3)

class SecureAgent:
    """Production-grade conversational agent utilizing PII redaction and secure memory storage."""
    def __init__(self, memory_store: ProductionMemoryStore) -> None:
        self.memory_store = memory_store
        self.llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a secure, helpful AI."),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ])
        
        chain = self.prompt | self.llm
        self.chain_with_memory = RunnableWithMessageHistory(
            chain,
            self.memory_store.get_history,
            input_messages_key="input",
            history_messages_key="chat_history",
        )
        
    def invoke(self, session_id: str, user_input: str) -> str:
        config = {"configurable": {"session_id": session_id}}
        try:
            response = self.chain_with_memory.invoke({"input": user_input}, config=config)
            return str(response.content)
        except Exception as e:
            logger.error(f"LLM invocation failed: {e}")
            return "I'm currently unable to process your request. Please try again later."

def advanced_memory_example() -> None:
    store = ProductionMemoryStore()
    agent = SecureAgent(store)
    
    session = "secure_user_99"
    input_text = "Hello, my email is confidential@example.com!"
    
    print(f"User: {input_text}")
    response = agent.invoke(session, input_text)
    print(f"Agent: {response}")
    
    history = store.get_history(session)
    print("\nVerifying Secure Memory Output (should be redacted):\n")
    for msg in history.messages:
        print(f"{type(msg).__name__}: {msg.content}")

if __name__ == "__main__":
    advanced_memory_example()


## Common Pitfalls in Production

1.  **Memory Leakage (Token Limits):** A long chat session will quickly exceed the context window. When using `ChatMessageHistory` in production, you must combine it with trimming techniques (like `trim_messages` in LangChain) or use a window buffer to avoid cost explosions and 'Lost in the Middle' hallucinations.
2.  **Stateless API Design:** If you are building an API (e.g., FastAPI), the memory object cannot be instantiated globally in Python memory. Every user/session needs its own memory instance retrieved from a database (like Redis) at the start of the request.
3.  **Summarization Latency:** Summarizing conversations requires additional LLM calls (`ConversationSummaryMemory` blocks execution during summarization). If your summarization LLM is slow, the user waits longer for their response. Consider doing summarization asynchronously.
4.  **Losing Fine Details:** If you summarize too aggressively using `ConversationSummaryMemory`, the model might forget specific names, IDs, or error codes mentioned earlier. A hybrid approach (Window + Summary) is often best.

## Practical Lab

**Your Task:**
Build a windowed memory buffer locally!
1. Initialize a `ConversationBufferWindowMemory` from `langchain.memory`.
2. Configure it with `k=3` (to retain only the last 3 interactions).
3. Insert 5 mock interactions using `save_context`.
4. Print the final buffer state to verify it strictly bounded the memory.
5. **Video Walkthrough:** Record a brief async video (e.g., using Loom) explaining your design decisions for configuring the buffer length and how you might persist this state in a database.


In [ ]:
from langchain.memory import ConversationBufferWindowMemory

def run_lab_implementation() -> None:
    """
    Executes the practical lab: setting up a bounded window memory and verifying it truncates older messages.
    """
    print("\n--- Lab Execution ---")
    
    # 1. Initialize Window Memory (k=3)
    lab_memory = ConversationBufferWindowMemory(k=3, memory_key="chat_history")
    
    # 2. Define 5 interactions
    interactions = [
        ("Hello", "Hi there"),
        ("My name is Charlie", "Nice to meet you, Charlie"),
        ("I like Python", "Python is great"),
        ("What's 2+2?", "4"),
        ("What did I say my name was?", "You said your name was Charlie")
    ]
    
    # 3. Save contexts
    for user_msg, ai_msg in interactions:
        lab_memory.save_context({"input": user_msg}, {"output": ai_msg})
    
    # 4. Print final buffer state
    print("Final Buffer State (Should only contain the last 3 interactions):")
    print(lab_memory.buffer)

run_lab_implementation()


## Reference Links
- [LangChain Memory Documentation](https://python.langchain.com/v0.2/docs/how_to/message_history/)
- [RunnableWithMessageHistory API Reference](https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html)
- [OWASP Top 10 for LLM Applications (For Security Insights)](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
